# LeafletFA Quickstart

This notebook demonstrates how to fit the LeafletFA model on a real single-cell splicing dataset.

**Dataset:** Mouse splicing foundation (Tabula Muris Senis + EasySci), hosted on Zenodo  
**DOI:** [10.5281/zenodo.18158125](https://doi.org/10.5281/zenodo.18158125)

**What this notebook covers:**
1. Download the dataset from Zenodo
2. Inspect the AnnData input format
3. Fit LeafletFA (K splicing programs)
4. Extract and interpret results
5. Visualize cell factor activities

---
**Installation:**
```bash
git clone https://github.com/daklab/LeafletFA.git
cd LeafletFA
pip install -e .
```

## 1. Download the dataset

In [ ]:
import urllib.request
import os

ZENODO_URL = (
    "https://zenodo.org/records/18158125/files/"
    "splice_adata_for_figures_mouse_foundation.h5ad?download=1"
)
LOCAL_PATH = "mouse_foundation_splicing.h5ad"

if not os.path.exists(LOCAL_PATH):
    print("Downloading dataset (~93 MB)...")
    urllib.request.urlretrieve(ZENODO_URL, LOCAL_PATH)
    print("Done.")
else:
    print("Dataset already downloaded.")

## 2. Inspect the AnnData input format

LeafletFA requires an [AnnData](https://anndata.readthedocs.io/) object with two specific layers produced by [ATSEmapper](https://github.com/daklab/ATSEmapper):

| Layer | Shape | Description |
|-------|-------|-------------|
| `cell_by_junction_matrix` | cells × junctions | Per-cell read counts for each splice junction |
| `cell_by_cluster_matrix` | cells × junctions | Total ATSE read counts (denominator for beta-binomial) |

Each column corresponds to one junction within an ATSE (Alternative Transcript Structure Event).  
The cluster matrix gives the total reads at all junctions in the same ATSE — this is what makes LeafletFA a proper compositional model.

In [ ]:
import anndata as ad
import numpy as np

adata = ad.read_h5ad(LOCAL_PATH)
print(adata)
print("\nLayers:", list(adata.layers.keys()))
print("\nJunction counts (cell_by_junction_matrix):")
print(" shape:", adata.layers["cell_by_junction_matrix"].shape)
print(" dtype:", adata.layers["cell_by_junction_matrix"].dtype)
print("\nATSE total counts (cell_by_cluster_matrix):")
print(" shape:", adata.layers["cell_by_cluster_matrix"].shape)
print("\nCell metadata columns:", list(adata.obs.columns))
print("\nJunction metadata columns:", list(adata.var.columns))

In [ ]:
# For this quickstart, subset to a manageable size
# (skip this cell if you want to train on the full dataset — needs GPU)
import scipy.sparse as sp

N_CELLS = 5000
N_JUNCTIONS = 10000

np.random.seed(42)
cell_idx = np.random.choice(adata.n_obs, size=min(N_CELLS, adata.n_obs), replace=False)

# Filter to junctions with enough coverage in the subset
adata_sub = adata[cell_idx, :].copy()
junc_counts = np.asarray(adata_sub.layers["cell_by_junction_matrix"].sum(axis=0)).flatten()
junc_idx = np.argsort(junc_counts)[::-1][:N_JUNCTIONS]
adata_sub = adata_sub[:, junc_idx].copy()

print(f"Subset: {adata_sub.n_obs} cells × {adata_sub.n_vars} junctions")
print(f"Sparsity: {1 - adata_sub.layers['cell_by_junction_matrix'].nnz / np.prod(adata_sub.layers['cell_by_junction_matrix'].shape):.1%} zeros")

## 3. Fit LeafletFA

The core workflow is four steps:
1. Initialize `LeafletFA(adata, K=...)` 
2. `from_anndata()` — convert to sparse PyTorch tensors
3. `train()` — stochastic variational inference
4. `get_all_variables()` — extract learned parameters

In [ ]:
from leafletfa import LeafletFA

K = 10  # number of splicing programs to learn

model = LeafletFA(
    adata_sub,
    K=K,
    waypoints_use=False,   # set True on the full dataset with PCA precomputed
    num_epochs=300,
    lr=0.01,
    log_wandb=False,
    loss_plot=True,
)

model.from_anndata()        # convert AnnData layers → sparse PyTorch tensors
model.train()               # fit via SVI
model.get_all_variables()   # extract psi, assign_post, pi, bb_conc

print("\n--- Results ---")
print(f"psi (splicing programs):     {model.psi.shape}  → K × junctions")
print(f"assign_post (cell activities): {model.assign_post.shape}  → cells × K")
print(f"pi (factor prevalences):     {model.pi.shape}")
print(f"\nFactor prevalences (pi): {np.asarray(model.pi).round(3)}")

## 4. Interpret splicing programs

`psi[k, j]` is the PSI (Percent Spliced In) of junction `j` in splicing program `k`.  
High PSI → junction is preferentially included in that program.  
Low PSI → junction is excluded.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

psi = np.asarray(model.psi)   # K × J
var = adata_sub.var

# For each splicing program, find the top 10 junctions by PSI deviation from mean
psi_centered = psi - psi.mean(axis=0, keepdims=True)

print("Top junctions per splicing program (by PSI deviation from mean):")
for k in range(K):
    top_junctions = np.argsort(psi_centered[k])[::-1][:5]
    genes = var.iloc[top_junctions]['gene_name'].values if 'gene_name' in var.columns else top_junctions
    print(f"  SP{k+1}: {genes}")

## 5. Visualize cell activities

`assign_post[c, k]` is cell `c`'s activity in splicing program `k`.  
These can be projected onto a UMAP or plotted against cell type annotations.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm

assign = np.asarray(model.assign_post)   # cells × K

# Plot cell activity distributions per factor
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for k, ax in enumerate(axes.flat):
    ax.hist(assign[:, k], bins=50, color=cm.tab10(k / K), edgecolor='none', alpha=0.8)
    ax.set_title(f"SP{k+1}  (π={model.pi[k]:.3f})", fontsize=9)
    ax.set_xlabel("Activity", fontsize=8)
    ax.set_ylabel("Cells", fontsize=8)
    ax.tick_params(labelsize=7)
plt.suptitle("Cell activity distributions per splicing program", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# If cell type annotations are available, show activity by cell type
if 'cell_type' in adata_sub.obs.columns or 'cell_ontology_class' in adata_sub.obs.columns:
    ct_col = 'cell_type' if 'cell_type' in adata_sub.obs.columns else 'cell_ontology_class'
    
    import seaborn as sns
    activity_df = pd.DataFrame(assign, columns=[f"SP{k+1}" for k in range(K)])
    activity_df[ct_col] = adata_sub.obs[ct_col].values
    
    # Show top 3 programs by variance
    top_programs = pd.Series(assign.var(axis=0), index=[f"SP{k+1}" for k in range(K)]).nlargest(3).index.tolist()
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for ax, sp in zip(axes, top_programs):
        ct_means = activity_df.groupby(ct_col)[sp].mean().sort_values(ascending=False).head(15)
        ct_means.plot(kind='barh', ax=ax, color='steelblue')
        ax.set_title(f"{sp} — mean activity by cell type")
        ax.set_xlabel("Mean activity")
    plt.tight_layout()
    plt.show()

## Next steps

**Differential splicing:** Use `leafletfa.differential_splicing` to compute ALBF scores and identify junctions that discriminate splicing programs.

**Running on your own data:** Use [ATSEmapper](https://github.com/daklab/ATSEmapper) to go from BAM files → regtools junction extraction → ATSEmapper → `SplicingDataset.h5ad`.

**Full dataset:** The complete mouse foundation dataset (all cells, all junctions) is available at [Zenodo 10.5281/zenodo.18158125](https://doi.org/10.5281/zenodo.18158125).

**GPU training:** For large datasets, use `enable_cpu_optimization=False` and call `model.initialize_triton_mask()` before `model.train()` (requires `pip install triton`).